In [1]:
# Imports and base setup
import os
import re
import json
import hashlib
from typing import Dict, List, Tuple

# Must be set BEFORE importing huggingface_hub/transformers
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Patch SSL verification BEFORE importing transformers/huggingface
import urllib3
urllib3.disable_warnings()

# Patch HTTPX to disable SSL verification
try:
    import httpx
    httpx._verify_disabled = True
except ImportError:
    pass

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder
from huggingface_hub import login

c:\projects\learn-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import faiss
import numpy as np
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


True

In [3]:
# Configure model names and initialize the LLM
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
# embedding_model_name = "BAAI/bge-base-en"

llm_model_name = "qwen/qwen3-32b"

hf_token = os.getenv("HF_TOKEN", "").strip().strip('"').strip("'")
if hf_token:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

llm = ChatGroq(
    model=llm_model_name,
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
 )

print(f"Embedding model: {embedding_model_name}")
print(f"LLM model: {llm_model_name}")

Embedding model: sentence-transformers/all-MiniLM-L6-v2
LLM model: qwen/qwen3-32b


In [4]:
# Load source text
candidates = [
    Path.cwd() / "data.txt",
    Path.cwd().parent / "data.txt",
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("data.txt not found in current directory or parent directory.")

text_data = data_path.read_text(encoding="utf-8")
print(f"Loaded {len(text_data)} characters from {data_path}")

Loaded 29819 characters from c:\projects\learn-rag\vectorDB\data.txt


In [5]:
# Clean source text before chunking
raw_text_data = text_data

# Remove inline reference markers and normalize formatting artifacts
cleaned_text = re.sub(r"\[(?:\d+|[a-z])\]", "", raw_text_data)
cleaned_text = cleaned_text.replace("\r\n", "\n").replace("\t", " | ")
cleaned_text = re.sub(r"[ \u00a0]+", " ", cleaned_text)
cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)

cleaned_lines = [line.strip() for line in cleaned_text.splitlines()]
cleaned_lines = [line for line in cleaned_lines if line]
text_data = "\n".join(cleaned_lines).strip()

print(f"Original length: {len(raw_text_data):,} characters")
print(f"Cleaned length: {len(text_data):,} characters")
print("Preview:")
print(text_data[:700])

Original length: 29,819 characters
Cleaned length: 28,932 characters
Preview:
Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl kəˈlɑːm/ ⓘ UB-duul kə-LAHM; 15 October 1931 – 27 July 2015) was an Indian aerospace scientist and statesman who served as the president of India from 2002 to 2007.
Born and raised in a Muslim family in Rameswaram, Tamil Nadu, Kalam studied physics and aerospace engineering. He spent the next four decades as a scientist and science administrator, mainly at the Defence Research and Development Organisation (DRDO) and Indian Space Research Organisation (ISRO) and was intimately involved in India's civilian space programme and military missile development efforts. He was known as the "Missile Man of India" for his work on the development of ballistic 


In [6]:
# Split text into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_text(text_data)
print(f"Created {len(chunks)} chunks")
print(chunks[0][:250])

Created 91 chunks
Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl kəˈlɑːm/ ⓘ UB-duul kə-LAHM; 15 October 1931 – 27 July 2015) was an Indian aerospace scientist and statesman who served as the president of India from 2002 to 2007.


In [7]:
chunks

['Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl kəˈlɑːm/ ⓘ UB-duul kə-LAHM; 15 October 1931 – 27 July 2015) was an Indian aerospace scientist and statesman who served as the president of India from 2002 to 2007.',
 "Born and raised in a Muslim family in Rameswaram, Tamil Nadu, Kalam studied physics and aerospace engineering. He spent the next four decades as a scientist and science administrator, mainly at the Defence Research and Development Organisation (DRDO) and Indian Space Research Organisation (ISRO) and was intimately involved in India's civilian space programme and military missile development efforts",
 '. He was known as the "Missile Man of India" for his work on the development of ballistic missile and launch vehicle technology. He also played a pivotal organisational, technical, and political role in Pokhran-II nuclear tests in 1998, India\'s second such test after the first test in 1974.',
 'Kalam was elected as the president of India in 2002 with the support of both the r

In [8]:
from langchain_text_splitters import SpacyTextSplitter

spacy_splitter = SpacyTextSplitter(
    pipeline="sentencizer",
    chunk_size=500,
    chunk_overlap=80,
)
spacy_chunks = spacy_splitter.split_text(text_data)

Created a chunk of size 618, which is longer than the specified 500


In [9]:
spacy_chunks

['Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl kəˈlɑːm/ ⓘ UB-duul kə-LAHM; 15 October 1931 – 27 July 2015) was an Indian aerospace scientist and statesman who served as the president of India from 2002 to 2007.\n\n\nBorn and raised in a Muslim family in Rameswaram, Tamil Nadu, Kalam studied physics and aerospace engineering.',
 'He spent the next four decades as a scientist and science administrator, mainly at the Defence Research and Development Organisation (DRDO) and Indian Space Research Organisation (ISRO) and was intimately involved in India\'s civilian space programme and military missile development efforts.\n\nHe was known as the "Missile Man of India" for his work on the development of ballistic missile and launch vehicle technology.',
 'He also played a pivotal organisational, technical, and political role in Pokhran-II nuclear tests in 1998, India\'s second such test after the first test in 1974.\n\n\nKalam was elected as the president of India in 2002 with the support of b

In [10]:
# Load a very lightweight Hugging Face embedding model
embedder = SentenceTransformer(embedding_model_name, device="cpu")

print(f"Loaded embedder: {embedding_model_name}")
print(f"Embedding dimension: {embedder.get_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2115.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedder: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


In [11]:
from langchain_core.embeddings import Embeddings

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model, query_prefix=""):
        self.model = model
        self.query_prefix = query_prefix

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).tolist()

    def embed_query(self, text):
        query_text = f"{self.query_prefix}{text}"
        return self.model.encode(
            [query_text],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )[0].tolist()

embedding_adapter = SentenceTransformerEmbeddings(
    embedder,
    query_prefix="Represent this sentence for searching relevant passages: ",
)


In [12]:
vectors = embedding_adapter.embed_documents(spacy_chunks)
print(f"Chunk vectors shape: {len(vectors)} x {len(vectors[0])}")

Chunk vectors shape: 69 x 384


In [13]:
vectors

[[-0.03241895139217377,
  0.0039011165499687195,
  -0.008546433411538601,
  0.053951092064380646,
  -0.05763204023241997,
  -0.0035480696242302656,
  0.05664543807506561,
  -0.02051173523068428,
  -0.031670473515987396,
  0.06507726013660431,
  -0.006562805734574795,
  -0.03401714935898781,
  -0.004510792903602123,
  0.008480851538479328,
  -0.02159682661294937,
  0.020724577829241753,
  -0.00752694858238101,
  0.06203383952379227,
  -0.028543775901198387,
  -0.06502452492713928,
  -0.06868041306734085,
  0.07543713599443436,
  0.017735948786139488,
  -0.06733179837465286,
  0.017418857663869858,
  0.0031039840541779995,
  0.08589458465576172,
  -0.03212951868772507,
  0.020278578624129295,
  0.0294013824313879,
  0.024881310760974884,
  -0.03662683442234993,
  -0.13961932063102722,
  0.05859539657831192,
  -0.08040757477283478,
  0.007662154268473387,
  -0.10627702623605728,
  0.10601265728473663,
  0.06558583676815033,
  -0.12635807693004608,
  0.04113706201314926,
  -0.0721923708915

In [14]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = embedder.get_embedding_dimension()
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embedding_adapter,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

print(f"Vector store initialized with dimension: {embedding_dim}")

Vector store initialized with dimension: 384


There are 2 approaches like we can precompute our embeddings and send to the VectorDB or we can directly send our details to the vectorDB it will use Sentence transformer and automatically embedd those documents. 

In [15]:
from uuid_utils import uuid4

uuids = [str(uuid4()) for _ in range(len(vectors))]
text_embeddings = list(zip(spacy_chunks, vectors))

vector_store.add_embeddings(text_embeddings=text_embeddings, ids=uuids)
print(f"Added {len(uuids)} embeddings to FAISS")

Added 69 embeddings to FAISS


In [28]:
# Save FAISS vector store to disk and reload for inference
store_dir = Path.cwd() / "faiss_store"
store_dir.mkdir(parents=True, exist_ok=True)

vector_store.save_local(folder_path=str(store_dir))
print(f"Saved vector store to: {store_dir}")

loaded_vector_store = FAISS.load_local(
    folder_path=str(store_dir),
    embeddings=embedding_adapter,
    allow_dangerous_deserialization=True,
 )

query = "where did APJ abdul kalam die ?"
results = loaded_vector_store.similarity_search(query, k=20)

print("Top 3 retrieved chunks from loaded store:")
answers = []
for i, doc in enumerate(results, start=1):
    answers.append(doc.page_content)
    print(f"\nResult {i}:\n{doc.page_content[:1000]}")

Saved vector store to: c:\projects\learn-rag\vectorDB\faiss_store
Top 3 retrieved chunks from loaded store:

Result 1:
Despite being placed in the intensive care unit, he was confirmed dead of a sudden cardiac arrest at 7:45 p.m. His purported last words to his aide Srijan Pal Singh were: "Funny guy!

Are you doing well?"


Aftermath
Dr. A. P. J. Abdul Kalam Memorial at Rameswaram
Following his death, the people of India paid tributes on social media.

The Government of India declared a seven-day state mourning period as a mark of respect.

Various leaders from India and abroad condoled the death of Kalam.

Result 2:
However, some of the locals were unconvinced by his statements on the safety of the plant, and were hostile to his visit.

In May 2012, Kalam launched a programme called What Can I Give Movement aimed at the youth of India with a central theme of defeating corruption.


Death
Main article: Death and state funeral of A. P. J. Abdul Kalam
On 27 July 2015, Kalam travelled to 

LOading the Re-ranker


In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Ensure Xet Storage is disabled (must also be set before huggingface_hub import in cell 3)
os.environ["HF_HUB_DISABLE_XET"] = "1"

hf_token = os.getenv("HF_TOKEN", "").strip().strip('"').strip("'")
token_arg = hf_token if hf_token else None

model = AutoModelForSequenceClassification.from_pretrained(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    # use_auth_token=token_arg
)
tokenizer = AutoTokenizer.from_pretrained(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    # use_auth_token=token_arg
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1537.99it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', token=token_arg)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7503.48it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
import torch

def top_k_rerank(question, answers, top_k=5):
    """Return top-k candidate answers ranked by cross-encoder relevance score."""
    if not answers:
        return []

    top_k = max(1, min(top_k, len(answers)))

    features = tokenizer(
        [question] * len(answers),
        answers,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    model.eval()
    with torch.no_grad():
        logits = model(**features).logits.squeeze(-1)

    scores = torch.sigmoid(logits).tolist()

    ranked = sorted(
        [{"answer": ans, "score": float(score)} for ans, score in zip(answers, scores)],
        key=lambda x: x["score"],
        reverse=True,
    )
    return ranked[:top_k]

# Example usage
question = "where did APJ abdul kalam die ?"
top_results = top_k_rerank(question, answers, top_k=10)
top_results

[{'answer': "Various leaders from India and abroad condoled the death of Kalam.\n\nKalam's body was flown to New Delhi on the morning of 28 July, where dignitaries including then president, vice president, and prime minister paid their last respects.\n\nHis body was placed in his Delhi residence for public viewing.\n\nOn 29 July, his body was flown to the town of Mandapam via Madurai, and was carried towards his home town of Rameswaram by road.",
  'score': 0.9991000890731812},
 {'answer': "He engaged in teaching, writing and public service after his presidency.\n\nHe was a recipient of several awards, including the Bharat Ratna, India's highest civilian honour.\n\n\nWhile delivering a lecture at IIM Shillong, Kalam collapsed and died from an apparent cardiac arrest on 27 July 2015, aged 83.\n\nThousands attended the funeral ceremony held in his hometown of Rameswaram, where he was buried with full state honours.\n\nA memorial was inaugurated near his home town in 2017.",
  'score': 0.

In [20]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [32]:
context_block = "\n\n".join(
    item["answer"] if isinstance(item, dict) and "answer" in item else str(item)
    for item in top_results
)

question = HumanMessage("where did APJ abdul kalam die ?")
system = SystemMessage(
    """You are a strict context-bound assistant. Use only the provided context. """
    """Do not use prior knowledge, outside facts, or assumptions. """
    """If the context does not explicitly contain the answer, say 'I cannot answer from the provided context.' """
    """Return exactly these 3 lines and nothing else:\n"""
    """answer: <answer or I cannot answer from the provided context.>\n"""
    """did you answer from the scope : <yes/no>\n"""
    """did it make any assumptions : <yes/no>\n\n"""
    f"""Context:\n{context_block}"""
)

messages = [system, question]
response = llm.invoke(messages)

print(response.content)

answer: APJ Abdul Kalam died in Shillong while delivering a lecture at IIM Shillong.  
did you answer from the scope : yes  
did it make any assumptions : no


In [37]:
import time

query = "where did APJ abdul kalam die ?"
timings = {}

pipeline_start = time.perf_counter()

load_start = time.perf_counter()
inference_vector_store = FAISS.load_local(
    folder_path=str(store_dir),
    embeddings=embedding_adapter,
    allow_dangerous_deserialization=True,
 )
timings["load_vectordb"] = time.perf_counter() - load_start

retrieval_start = time.perf_counter()
retrieved_docs = inference_vector_store.similarity_search(query, k=20)
retrieved_answers = [doc.page_content for doc in retrieved_docs]
timings["results"] = time.perf_counter() - retrieval_start

rerank_start = time.perf_counter()
reranked_results = top_k_rerank(query, retrieved_answers, top_k=10)
timings["reranking"] = time.perf_counter() - rerank_start

context_block = "\n\n".join(
    item["answer"] if isinstance(item, dict) and "answer" in item else str(item)
    for item in reranked_results
)

messages = [
    SystemMessage(
        """You are a strict context-bound assistant. Use only the provided context. """
        """Do not use prior knowledge, outside facts, or assumptions. """
        """If the context does not explicitly contain the answer, say 'I cannot answer from the provided context.' """
        """Return exactly these 3 lines and nothing else:\n"""
        """answer: <answer or I cannot answer from the provided context.>\n"""
        """did you answer from the scope : <yes/no>\n"""
        """did it make any assumptions : <yes/no>\n\n"""
        f"""Context:\n{context_block}"""
    ),
    HumanMessage(query),
]

llm_start = time.perf_counter()
streamed_parts = []
print("Output:\n")
for chunk in llm.stream(messages):
    chunk_text = chunk.content if isinstance(chunk.content, str) else ""
    if chunk_text:
        streamed_parts.append(chunk_text)
        print(chunk_text, end="", flush=True)
print()
timings["llm_call"] = time.perf_counter() - llm_start

inference_response = "".join(streamed_parts)
timings["total"] = time.perf_counter() - pipeline_start

print(f"\nQuery: {query}")
print(f"Vector DB load time: {timings['load_vectordb']:.4f} seconds")
print(f"Retrieval time: {timings['results']:.4f} seconds")
print(f"Reranking time: {timings['reranking']:.4f} seconds")
print(f"LLM call time: {timings['llm_call']:.4f} seconds")
print(f"Total pipeline time: {timings['total']:.4f} seconds")

Output:

answer: A.P.J. Abdul Kalam died in Shillong while delivering a lecture at IIM Shillong.  
did you answer from the scope : yes  
did it make any assumptions : no

Query: where did APJ abdul kalam die ?
Vector DB load time: 0.0020 seconds
Retrieval time: 0.0258 seconds
Reranking time: 0.8691 seconds
LLM call time: 0.8355 seconds
Total pipeline time: 1.7333 seconds


## Diagnostics: Why Context Retrieval is Poor

Let's analyze the chunking and retrieval issues

In [22]:
# Problem 1: Analyze Chunk Content and Structure
print("=" * 80)
print("PROBLEM 1: CHUNK ANALYSIS - Are chunks breaking semantic meaning?")
print("=" * 80)

print(f"\nTotal chunks created: {len(chunks)}")
print(f"Average chunk size: {sum(len(c) for c in chunks) / len(chunks):.0f} characters")
print(f"Min chunk size: {min(len(c) for c in chunks)}, Max chunk size: {max(len(c) for c in chunks)}")

# Show first 5 chunks to see how they're being split
print("\nFirst 5 chunks:")
for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i}:")
    print(f"Length: {len(chunk)} chars")
    print(f"Content: {chunk[:200]}...")
    if "2016" in chunk or "Sunrisers" in chunk:
        print("⚠️  Contains relevant keywords!")


PROBLEM 1: CHUNK ANALYSIS - Are chunks breaking semantic meaning?

Total chunks created: 91
Average chunk size: 325 characters
Min chunk size: 29, Max chunk size: 499

First 5 chunks:

Chunk 0:
Length: 208 chars
Content: Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl kəˈlɑːm/ ⓘ UB-duul kə-LAHM; 15 October 1931 – 27 July 2015) was an Indian aerospace scientist and statesman who served as the president of India from 2002 ...

Chunk 1:
Length: 401 chars
Content: Born and raised in a Muslim family in Rameswaram, Tamil Nadu, Kalam studied physics and aerospace engineering. He spent the next four decades as a scientist and science administrator, mainly at the De...

Chunk 2:
Length: 291 chars
Content: . He was known as the "Missile Man of India" for his work on the development of ballistic missile and launch vehicle technology. He also played a pivotal organisational, technical, and political role ...

Chunk 3:
Length: 387 chars
Content: Kalam was elected as the president of India in 2002 with

In [23]:
# Problem 2: Check Retrieved Results Quality
print("\n" + "=" * 80)
print("PROBLEM 2: RETRIEVAL QUALITY - What is actually being returned?")
print("=" * 80)

test_query = "when did SRH win their first IPL title"
search_results = loaded_vector_store.similarity_search(test_query, k=10)

print(f"\nQuery: '{test_query}'")
print(f"Retrieved {len(search_results)} results:\n")

for i, doc in enumerate(search_results, 1):
    print(f"\n--- Result {i} ---")
    print(f"Content: {doc.page_content[:300]}")
    # Check if result contains the actual answer
    if "2016" in doc.page_content and "Sunrisers" in doc.page_content:
        print("✅ Contains the answer!")
    elif "Sunrisers Hyderabad" in doc.page_content:
        print("⚠️  Contains team name but might lack context")
    else:
        print("❌ Doesn't appear to contain relevant answer")



PROBLEM 2: RETRIEVAL QUALITY - What is actually being returned?

Query: 'when did SRH win their first IPL title'
Retrieved 10 results:


--- Result 1 ---
Content: ISBN 978-8-125-04212-9.


A. P. J. Abdul Kalam; Srijan Pal Singh (2011).

Target 3 Billion: Innovative Solutions towards Sustainable Development.

Penguin Books.

ISBN 978-0-143-41730-9.


A. P. J. Abdul Kalam; Poonam Kohli (2012).

You are Unique: Scale New Heights by Thoughts and Actions.

Punya P
❌ Doesn't appear to contain relevant answer

--- Result 2 ---
Content: No manoeuvres are required any more, as I am placed in my final position in eternity."


Writings
Main article: A. P. J. Abdul Kalam bibliography
Kalam delivering a speech in 2010
Kalam has authored various books during his career, and his books have garnered interest in various countries.


In his 
❌ Doesn't appear to contain relevant answer

--- Result 3 ---
Content: We Can do it: Thoughts for Change.

Shree Book Centre.

ISBN 978-9-350-49763-0.


A. P. J. A

In [24]:
# Problem 3: Compare Embedding Models and Chunking Strategies
print("\n" + "=" * 80)
print("PROBLEM 3: EMBEDDING + CHUNKING STRATEGY MISMATCH")
print("=" * 80)

# Compare chunking approaches
print("\nRecursive Character Splitter chunks:")
print(f"  - Total chunks: {len(chunks)}")
print(f"  - Avg size: {sum(len(c) for c in chunks) / len(chunks):.0f} chars")

print("\nSpacy Splitter chunks (sentence-based):")
print(f"  - Total chunks: {len(spacy_chunks)}")
print(f"  - Avg size: {sum(len(c) for c in spacy_chunks) / len(spacy_chunks):.0f} chars")

# Check if the issue is with table/structured data
print("\n\nAnalyzing data structure:")
lines = text_data.split('\n')
table_lines = [l for l in lines if '\t' in l or '|' in l or any(c.isdigit() for c in l)]
print(f"  - Total lines: {len(lines)}")
print(f"  - Lines with table-like structure: {len(table_lines)}")

if len(table_lines) > 0:
    print("\n⚠️  ISSUE FOUND: Document contains structured data (tables)")
    print("    Small chunks (500 chars) may be breaking up table rows!")
    print("\nExample table content:")
    for line in table_lines[:5]:
        print(f"    {line[:100]}")



PROBLEM 3: EMBEDDING + CHUNKING STRATEGY MISMATCH

Recursive Character Splitter chunks:
  - Total chunks: 91
  - Avg size: 325 chars

Spacy Splitter chunks (sentence-based):
  - Total chunks: 69
  - Avg size: 441 chars


Analyzing data structure:
  - Total lines: 111
  - Lines with table-like structure: 73

⚠️  ISSUE FOUND: Document contains structured data (tables)
    Small chunks (500 chars) may be breaking up table rows!

Example table content:
    Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl kəˈlɑːm/ ⓘ UB-duul kə-LAHM; 15 October 1931 – 27 July 2
    Born and raised in a Muslim family in Rameswaram, Tamil Nadu, Kalam studied physics and aerospace en
    Kalam was elected as the president of India in 2002 with the support of both the ruling Bharatiya Ja
    While delivering a lecture at IIM Shillong, Kalam collapsed and died from an apparent cardiac arrest
    Avul Pakir Jainulabdeen Abdul Kalam was born on 15 October 1931 to a Tamil Muslim family in the pilg


In [25]:
# Problem 4: Proposed Solutions
print("\n" + "=" * 80)
print("SOLUTIONS TO IMPROVE CONTEXT CAPTURE")
print("=" * 80)

print("""
ROOT CAUSES:
1. ❌ Chunk Size (500 chars): Too small for structured data with context
   - Table rows are split across multiple chunks
   - Semantic relationships are lost
   
2. ❌ Separator Strategy: Breaks on any space when preferred text unavailable
   - Doesn't understand document structure
   - Loses context around important sections
   
3. ❌ Embedding Model: All-MiniLM is good but not domain-optimized
   - May not capture cricket/sports terminology effectively
   
4. ❌ No Context Awareness: Splitter doesn't understand tables/lists
   - Winners table gets fragmented

RECOMMENDATIONS:

Option A: INCREASE CHUNK SIZE (Recommended for this use case)
  - Increase chunk_size from 500 to 1000-1500
  - Keep overlap at 200-300
  - Reason: This is structured data with tables; larger chunks preserve context

Option B: USE SEMANTIC CHUNKING
  - Use sentence-based chunking (Spacy) instead
  - Better for narrative + structured data mix
  - Already available in notebook

Option C: HYBRID APPROACH (Best)
  - Use larger chunks: 1000 chars
  - Add chunk metadata to preserve document structure
  - Consider re-ranking retrieved results
  
Option D: BETTER EMBEDDING MODEL
  - Use more specialized model: all-mpnet-base-v2 (larger)
  - Or use: all-roberta-large-v1 (better semantic understanding)
  - Trade-off: Slower but better accuracy
""")



SOLUTIONS TO IMPROVE CONTEXT CAPTURE

ROOT CAUSES:
1. ❌ Chunk Size (500 chars): Too small for structured data with context
   - Table rows are split across multiple chunks
   - Semantic relationships are lost

2. ❌ Separator Strategy: Breaks on any space when preferred text unavailable
   - Doesn't understand document structure
   - Loses context around important sections

3. ❌ Embedding Model: All-MiniLM is good but not domain-optimized
   - May not capture cricket/sports terminology effectively

4. ❌ No Context Awareness: Splitter doesn't understand tables/lists
   - Winners table gets fragmented

RECOMMENDATIONS:

Option A: INCREASE CHUNK SIZE (Recommended for this use case)
  - Increase chunk_size from 500 to 1000-1500
  - Keep overlap at 200-300
  - Reason: This is structured data with tables; larger chunks preserve context

Option B: USE SEMANTIC CHUNKING
  - Use sentence-based chunking (Spacy) instead
  - Better for narrative + structured data mix
  - Already available in noteb

In [26]:
# SOLUTION 1: Improve with Larger Chunk Size
print("\n" + "=" * 80)
print("SOLUTION 1: LARGER CHUNK SIZE (Quick Fix)")
print("=" * 80)

# Create improved chunker with larger chunks
improved_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,  # Increased from 500
    chunk_overlap=300,  # Increased from 100
    separators=["\n\n", "\n", ". ", " ", ""],
)

improved_chunks = improved_splitter.split_text(text_data)
print(f"\nImproved chunks: {len(improved_chunks)}")
print(f"Avg size: {sum(len(c) for c in improved_chunks) / len(improved_chunks):.0f} chars")
print(f"Size reduction: {len(chunks)} → {len(improved_chunks)} chunks")

# Create new vector store with improved chunks
improved_vectors = embedding_adapter.embed_documents(improved_chunks)
improved_uuids = [str(uuid4()) for _ in range(len(improved_vectors))]

improved_index = faiss.IndexFlatL2(embedding_dim)
improved_vector_store = FAISS(
    embedding_function=embedding_adapter,
    index=improved_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

improved_text_embeddings = list(zip(improved_chunks, improved_vectors))
improved_vector_store.add_embeddings(text_embeddings=improved_text_embeddings, ids=improved_uuids)

# Test the improved retrieval
print("\n\nTesting improved retrieval:")
query = "when did SRH win their first IPL title"
improved_results = improved_vector_store.similarity_search(query, k=3)

for i, doc in enumerate(improved_results, 1):
    print(f"\nResult {i}:")
    print(f"Content: {doc.page_content[:400]}")



SOLUTION 1: LARGER CHUNK SIZE (Quick Fix)

Improved chunks: 32
Avg size: 973 chars
Size reduction: 91 → 32 chunks


Testing improved retrieval:

Result 1:
Content: A. P. J. Abdul Kalam; Y. S. Rajan (2011). The Scientific India: A Twenty First Century Guide to the World around Us. Penguin Books. ISBN 978-0-143-41687-6.
A. P. J. Abdul Kalam; Arun Tiwari (2011). Failure to Success: Legendry Lives. Orient Blackswan. ISBN 978-8-125-04212-9.
A. P. J. Abdul Kalam; Srijan Pal Singh (2011). Target 3 Billion: Innovative Solutions towards Sustainable Development. Pengu

Result 2:
Content: In 1969, Kalam transferred to ISRO where he became the project director of India's first satellite launch vehicle (SLV) which successfully deployed the Rohini satellite in near-earth orbit in July 1980. He had earlier started work on an expandable rocket project independently at DRDO in 1965. In 1969, Kalam received the approval from the Government of India to expand the programme to include more 

Result 3:
Co